# Qwen-Image 2.1 — the MegaLiminal LoRA on Colab (thin shell over the repo runner)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/qwen-image21/notebooks/qwen_image21_mega_liminal.ipynb)

Trains a LoRA for **[Qwen-Image 2.1](https://huggingface.co/Qwen/Qwen-Image-2.1)** (text to image) on
**[AbstractPhil/mega-liminal](https://huggingface.co/datasets/AbstractPhil/mega-liminal)** (1,873 captioned images of
empty, in-between places in ten class folders) through model type `qwen_image21` of the
[AbstractEyes diffusion-pipe fork](https://github.com/AbstractEyes/diffusion-pipe) (release `qwen-image21-v1`), and uploads
every LoRA, preview and measurement into the folder `qwen_image21/` of
**[AbstractPhil/mega-liminal-lora](https://huggingface.co/AbstractPhil/mega-liminal-lora)** as it goes.

**Why a screen comes before the run.** Qwen-Image 2.1 has its guidance trained in: it samples at cfg 1. Ordinary
training wears that guidance away, so the longer a LoRA trains, the more it needs real CFG at inference. The fork can
instead train against the model's own guided output at a strength w (the guided loss), which keeps cfg 1 working. The
strength trained into 2.1 is not published, so `q.screen()` trains four short runs (the plain loss, and the guided loss
at w = 3, 4 and 5; 400 steps each) and measures each one before and after training on 32 ordinary prompts at two seeds
(PartiPrompts) and on eight of the LoRA's own prompts, as picture distances (DINOv2):

| measurement | what it is |
|---|---|
| ordinary prompts changed | same prompt, same seed: the picture before training against the picture after |
| seed-to-seed change | same prompt before training: one seed against another (the yardstick) |
| untrained model rendered again | each later run's first previews against the first run's (must stay under the yardstick, or nothing is picked) |
| own subject changed | the eight liminal prompts, before against after |
| contrast kept | the pictures' luminance spread after training over before, on the ordinary prompts |

A run is eligible when its ordinary prompts change less than a seed change and keep at least 90% of their contrast;
of the eligible runs, the one that changes its own subject most per unit of change elsewhere is picked, and
`q.train()` trains with it (4 epochs, a LoRA and fifteen previews at every epoch).

**Same thin-shell contract as the other notebooks:** all logic is in the repo
(`geolip_anima_trainer.qwen_image21_runner.QwenImage21Runner`); this notebook is clone → install → a few `q.<step>()`
calls. **To change anything, edit the repo and `git pull`; the notebook stays as it is.** Every long step prints the
time spent and the time left as it goes.

**Before you start:** an RTX PRO 6000 runtime, and a **write** `HF_TOKEN` in Colab Secrets (🔑) with notebook access:
the uploads need it, and `screen()` and `train()` stop before any training if they cannot write.

> **Licences.** Qwen-Image 2.1 is under the Qwen Research License Agreement: non-commercial use only (research or
> evaluation). A LoRA trained on it is a derivative and carries the same terms ("Built with Qwen"); a copy of the
> licence and its NOTICE are uploaded beside the LoRAs. MegaLiminal's pictures belong to their creators and are shared
> for research and non-commercial use (see its card). DINOv2 (the screen's picture distance) is Apache-2.0.

## 0 · Confirm the GPU + disk

About 60 GB of disk: the three model files (24 GB), the dependencies, the dataset and its cache, the LoRAs and the
training checkpoints.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap — clone + install (→ restart ONCE the first time)

Clones this repo (the `qwen-image21` branch, which carries the 2.1 runner) and the diffusion-pipe fork at its release
`qwen-image21-v1` (into `external/diffusion-pipe-fork-qwen-image21-v1`, beside any checkout the Anima and Sana notebooks
use), and installs the dependencies (the logic is in `anima_colab.install`, so a `git pull` updates it with no notebook
edit). **Run the cell; if the kernel restarts, run it again** (it then skips), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
BRANCH = "qwen-image21"                  # the branch that carries the Qwen-Image 2.1 runner
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)   # the latest runner before importing it
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, dp_ref=anima_colab.QWEN21_FORK_REF, similarity=False):
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after the restart) Setup and the parity check

`setup()` checks the GPU, logs in, finds the fork and downloads the three model files of
[Comfy-Org/Qwen-Image-2.1](https://huggingface.co/Comfy-Org/Qwen-Image-2.1) (the bf16 transformer, the VAE and the
Qwen3-VL-8B text encoder). `parity()` runs the fork's own check on this card: the trainer's text encoder, VAE and
transformer against ComfyUI's, the padding of captions of different lengths, the preview sampler, and one LoRA step
with the plain and the guided loss. It prints one PASS / FAIL row per check, uploads its log to `qwen_image21/checks/`,
and **stops on any FAIL** (then nothing below should run).

In [ ]:
from geolip_anima_trainer.qwen_image21_runner import QwenImage21Runner
q = QwenImage21Runner()   # uploads to AbstractPhil/mega-liminal-lora/qwen_image21 (release_repo="you/repo" or None to change)
q.setup()                 # env + HF login + GPU check + the fork + the model files (~24 GB the first time)

In [ ]:
q.parity()                # the trainer computes what ComfyUI computes, on this card; stops on any FAIL

## 3 · The dataset and the screen

`prepare_dataset()` downloads MegaLiminal (one folder of pictures and `.txt` captions per class) and sets the repeats
per class (square-root dampened, at most 8 per picture): 2,789 samples an epoch. `screen()` trains and reads the four
short runs described at the top and prints the table and the pick; each run's LoRA, previews, before / after sheet and
measurements upload to `qwen_image21/screen/` as they finish. **A rerun reuses every run already measured with the same
settings** (here, or uploaded by an earlier runtime), so after a disconnect it picks up where it stopped.

In [ ]:
q.prepare_dataset()       # the ten classes, their pictures and repeats

In [ ]:
q.screen()                # four short runs + their measurements -> the table and the pick

## 4 · The full run

`train()` trains with the screen's pick (to choose yourself: `q.train(cfg=4.0)` for the guided loss at w = 4, or
`q.train(cfg=None)` for the plain loss): rank 32, plain Adam at 1e-4 without weight decay, 4 epochs, a save and fifteen
previews every epoch. Each epoch's LoRA and previews upload to `qwen_image21/rank32/` once the next save exists; the
last one, the configs, the log and the TensorBoard events at the end. Uploads retry in the background and never stop
the run.

In [ ]:
q.train()                 # the screen's pick; the trainer's log, the step and the time left print here

## 5 · Contact sheets

In [ ]:
q.sheets()                # one sheet per save (five previews a row), uploaded beside the LoRAs

## Notes

- **Where things land:** `AbstractPhil/mega-liminal-lora`, folder `qwen_image21/`: `README.md`, `LICENSE`, `NOTICE`;
  `rank32/` (`mega-liminal-qwen-image21-r32-<loss>-eNN.safetensors`, `adapter_config.json`, `samples/`, `training/`);
  `screen/` (the four short runs, `reads/`, `screen.md`); `checks/` (the parity log). `<loss>` is `plain` or `w3` /
  `w4` / `w5`. Locally: `/content/qwen_image21_data/`.
- **After a disconnect or on a new runtime:** run the bootstrap, `setup()` and `prepare_dataset()` again. `screen()`
  reuses the measured runs; `train()` starts over on a new runtime (the LoRAs already uploaded stay). `q.status()`
  lists runs, saves and anything not uploaded yet; `q.tail()` shows the end of a log.
- **Changing the recipe:** `QwenImage21Runner(rank=64, epochs=6, lr=5e-5)`; an attention-only LoRA:
  `exclude_modules=["img_mlp.gate_up", "img_mlp.out"]`. Changed settings measure a fresh screen.
- **Using a LoRA:** ComfyUI's **LoraLoaderModelOnly** in the Qwen-Image 2.1 workflow (the base transformer, 25 steps,
  cfg 1). The previews are rendered the same way (euler, the model's own time shift).
- **Publishing:** the LoRAs carry Qwen's research licence: non-commercial listings only, with "Built with Qwen".
- **Updating logic = `git pull`** (the bootstrap cell pulls every run); nothing here needs editing.